## Keypoint Extraction
Extracts MediaPipe pose keypoints from video clips for 6 shot classes.

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import os
from tqdm import tqdm

In [2]:
# ── Shot labels (6 classes) ────────────────────────────────────────────
LABEL_NAMES = [
    'Cover Drive',
    'Cut',
    'Flick',
    'Forward Defence',
    'Pull',
    'Sweep',
]

mp_pose = mp.solutions.pose
pose    = mp_pose.Pose(static_image_mode=False)

In [3]:
def extract_keypoints(video_path):
    """
    Extract raw MediaPipe keypoints from every frame.
    Returns array of shape (T, 99) — 33 landmarks × 3 coords (x, y, z).
    Frames where pose detection fails are skipped.
    """
    cap    = cv2.VideoCapture(video_path)
    frames = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results   = pose.process(frame_rgb)

        if results.pose_landmarks:
            kp = []
            for lm in results.pose_landmarks.landmark:
                kp.extend([lm.x, lm.y, lm.z])
            frames.append(kp)

    cap.release()
    return np.array(frames)  # (T, 99)

In [4]:
def sample_frames(sequence, target_len=100):
    """Uniformly sample target_len frames from a sequence (handles short clips)."""
    if len(sequence) == 0:
        return np.zeros((target_len, 99))
    idx = np.linspace(0, len(sequence) - 1, target_len).astype(int)
    return sequence[idx]

In [5]:
def normalize_pose(sequence):
    """
    Canonical pose normalisation:
      - Translate so hip midpoint = origin
      - Scale by left torso segment (shoulder-11 to hip-23 distance)
    Returns shape (T, 99).
    """
    seq = sequence.copy().reshape(-1, 33, 3)

    for i in range(len(seq)):
        kp         = seq[i]
        hip_center = (kp[23] + kp[24]) / 2
        kp         = kp - hip_center
        torso      = np.linalg.norm(kp[11] - kp[23]) + 1e-6
        kp         = kp / torso
        seq[i]     = kp

    return seq.reshape(-1, 99)

In [6]:
def normalize_pose(sequence):
    """
    Canonical pose normalisation:
      - Translate so hip midpoint = origin
      - Scale by left torso segment (shoulder-11 to hip-23 distance)
    Returns shape (T, 99).
    """
    seq = sequence.copy().reshape(-1, 33, 3)

    for i in range(len(seq)):
        kp         = seq[i]
        hip_center = (kp[23] + kp[24]) / 2
        kp         = kp - hip_center
        torso      = np.linalg.norm(kp[11] - kp[23]) + 1e-6
        kp         = kp / torso
        seq[i]     = kp

    return seq.reshape(-1, 99)

In [7]:
def mirror_keypoints_flat(sequence):
    """
    Mirror a left-handed batter to canonical right-handed frame.
    Flips x-axis and swaps left/right landmark pairs in-place.
    sequence: (T, 99)
    """
    seq = sequence.copy().reshape(-1, 33, 3)
    seq[:, :, 0] = -seq[:, :, 0]   # flip x

    swap_pairs = [
        (11, 12),   # shoulders
        (13, 14),   # elbows
        (15, 16),   # wrists
        (23, 24),   # hips
        (25, 26),   # knees
        (27, 28),   # ankles
    ]
    for l, r in swap_pairs:
        seq[:, [l, r]] = seq[:, [r, l]]

    return seq.reshape(-1, 99)


def detect_handedness(sequence):
    """
    Infer batting hand from the mid-sequence wrist positions.
    For a right-handed batter the right wrist (16) tends to be lower (higher y)
    than the left at impact.
    Returns 'right' or 'left'.
    """
    seq = sequence.reshape(-1, 33, 3)
    mid = len(seq) // 2
    window = seq[max(0, mid - 5): mid + 5]
    l_y = np.mean(window[:, 15, 1])
    r_y = np.mean(window[:, 16, 1])
    return 'right' if r_y >= l_y else 'left'

In [8]:
def augment(sequence):
    """
    Light augmentation — preserves biomechanical structure:
      1. Gaussian noise  (σ = 0.01, matches normalised scale)
      2. Temporal jitter (±5 frames, 50% probability)
      3. Mirror augmentation for right-handed clips (30% probability)
         produces a synthetic left-handed variant which then gets
         re-mirrored to canonical frame — net effect is subtle style variation.
    """
    seq = sequence.copy()

    # 1. Noise
    seq += np.random.normal(0, 0.01, seq.shape)

    # 2. Temporal jitter
    if np.random.rand() > 0.5:
        shift = np.random.randint(-5, 5)
        seq   = np.roll(seq, shift, axis=0)

    # 3. Mirror augmentation (creates subtle style variation)
    if np.random.rand() < 0.30:
        seq = mirror_keypoints_flat(seq)

    return seq

In [9]:
DATA_DIR  = r"D:\CA_POC\dataset"

# Build label map from folder names; sort to ensure deterministic ordering
labels    = [l for l in sorted(os.listdir(DATA_DIR)) if l != 'Backward Defence']
label_map = {label: i for i, label in enumerate(labels)}

print("Shot classes found:")
for label, idx in label_map.items():
    print(f"  {idx}: {label}")

X, y = [], []

Shot classes found:
  0: Cover Drive
  1: Cut
  2: Flick
  3: Forward Defense
  4: Pull
  5: Sweep


In [10]:
for label in labels:
    folder = os.path.join(DATA_DIR, label)

    for file in tqdm(os.listdir(folder), desc=label):
        path = os.path.join(folder, file)

        try:
            kp = extract_keypoints(path)

            if len(kp) < 10:
                continue

            kp = sample_frames(kp, 100)
            kp = normalize_pose(kp)

            # ── Canonical right-handed frame ───────────────────────────
            # Mirror left-handed batters before storing so all downstream
            # logic (features, ideal poses, rules) uses one consistent frame.
            if detect_handedness(kp) == 'left':
                kp = mirror_keypoints_flat(kp)

            # Original
            X.append(kp)
            y.append(label_map[label])

            # Augmented ×1
            X.append(augment(kp))
            y.append(label_map[label])

        except Exception as e:
            print(f"Error in {file}: {e}")

Sweep: 100%|██████████| 77/77 [02:34<00:00,  2.01s/it]


In [11]:
X = np.array(X)
y = np.array(y)

print("Final Shape:", X.shape, y.shape)

from collections import Counter
print("Class distribution:", Counter(y.tolist()))

os.makedirs(r"D:\CA_POC\new_processed", exist_ok=True)
np.save(r"D:\CA_POC\new_processed\X.npy", X)
np.save(r"D:\CA_POC\new_processed\y.npy", y)
np.save(r"D:\CA_POC\new_processed\label_map.npy", label_map)

Final Shape: (976, 100, 99) (976,)
Class distribution: Counter({0: 170, 4: 170, 3: 166, 1: 160, 2: 156, 5: 154})
